# 24. Monthly active users are not the sum of the days

**Track:** sql | **Difficulty:** medium | **Tags:** distinct count, active users, grain, aggregation

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("024-monthly-active-users")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "logins": """
UEFSMRUEFfAGFdYDTBVuFQASAAC4AwQWAAkBABkJBwQAEg0IACUNCAAEDQgANg0IABgNCAAnDQgAIQ0IACwNCAABDQgACQ0IADUN
CAAqDQgAEA0IAAwNCAAfDQgAEQ0IAAYNCAAiDQgACw0IACQNCAATDQgAMQ0IACANCAADDQgANA0IAA4NCAAmDQgALw0IABwNCAAb
DQgADw0IADcNCAApDQgADQ0IAAcNCAAwDQgAAg0IABUNCAAjDQgAFA0IADMNCAQdAAkBADIJBwQAGg0IAC4NCAAtDQgAHg0IACsN
CAAKDQgAKA0IAAgNCDwXAAAAAAAAAAUAAAAAAAAAFQAVuAYVwgYsFaQIFRAVBhUGHBgINwAAAAAAAAAYCAEAAAAAAAAAFgAoCDcA
AAAAAAAAGAgBAAAAAAAAABERAAAAnAP0mwEDAAAApAgBBn9AIAxEYRxGcCBJoCxM4zwOFDEQMUhTRBUOoVSGBV1Ypm1c53gf1IGh
6C1K5GjZxA0jMwUENI0YKZWmASwC9EgnWgLdk4YhmW6DiioK+FRk5BSNAoVqoGZKph4euGpj6LHMoCzLahHfAB3KhkqGxmRO+AWt
knXV8lxONrksFlylwRzuK1qkOaifF2WK1hXwCqjqO1RftIiQARfTwabf8BpWyRwcx6kxyphPPBkQ2HxMCqrLVTzNoT6yka2vFShT
sMQyBn1GI5gz9ETmNNHy9y0pUtBMgaRYWoFDHbOGckAM1GXpqsqG2KRKENEjtCqw6zFZMLlNRkZNJKniBDEPyJjK1DT053THYqG2
pTwDU4/Aa2SGknIPESxr4MzES66DDQEfmoEX8izG6gHO3AZQBVBfw47wGIjH4UzByBWnYQbc0kzd8LLLoiLFdKWVqaovJHp280bE
Qj/yN6ePrLJK6hkqSqzscLxDRTdWgwIQQ6gDRhvMgAqpSlcYYS0JJcJqqMt5kCijdGVXjQgdnFCWHgEAAAAAFQQVMhU2TBUGFQAS
AAAZYAcAAABhbmRyb2lkAwAAAGlvcwMAAAB3ZWIVABWgAhWoAiwVpAgVEBUGFQYcNgAoA3dlYhgHYW5kcm9pZBERAAAAkAHwjwMA
AACkCAECfwRlFZaZVAVVFFUIhBYmBQZWlSSIJQCZFkQiRZKVWCIqkVpShpFoYWAZUFRYQZpRSglmZUViUUASWqFFlJamFqWSCmaU
qpFFBCBRlZlllGZiFlpGVVKkShmElKgVloaZECIhKVhBilpVSURoIRBVUJZoqWoapZGQoAlVYEEllAkAgCUCVgQGABUEFaBCFZxA
TBWkCBUAEgAAkCHw9cC+0Ma6LQYAgPA/lvIuBgDAgAYWCS8GAMDsEwCjMQYAAMDLSHUtBgBAI/x3xDAGAEBzECoPLwYAQDs1Yg0v
BgCA3H7NGC4GAIBm7PmWLAYAwCzba04xBgBAh+gFyy0GAEBPmEi6LgYAwEvuuBsrBgDATf5OtSoGAIBa1fQMLwYAQNwk9/0sBgAA
2J+kSy0GAID4hzrpKgYAABGLnmErBgBAWKwHEywGAIB4xRlvKwYAQEBYyWgwBgAArxzwvS8GAEByFUS0KgYAgGCq6vouBgBAGKPE
SjAGAACJ/bOdMAYAQH4oQxcxBgCAvwkFCS8GAMDyfFNPLAHgiLMB1TAGAMB+fP4OLAYAwOQtxqkqBgBApZnHEywGAEBepqwDAfj0
UwHROj8SLwYAAFNUCXAuBgAA4Ul7+y8GAMAfbveIKwYAAFfDLIotBgBAUxFSXC4GAIAIauYaLgYAQGSJ5K8tBgCAO1OVTSsGAMB+
aKcYLgYAwJJgJNwsBgDAPSPSCS8GAICoclQwLwYAAAP6DG4sBgDAkVtd4y4GAEBCK3Y/LQYAgEtFKsUrBgBAaPNlqy4GAIAnQfiW
LAYAgFDii+0sBgBABdIgoSoGAICtl1nWLAYAQETkOJErBgDA2lLW9SsGAECT6AntLQYAwAltjB4xBgAA7bp2qyoGAIBhIJO0LQYA
wHekOdUsBgAA+OdwJzEGAICyQ8H7LwYAwByKMGsuBgAA4SIXSisGAIAPB4B9LgYAgB6eGv8vBgDA3SHBiSsGAIDyYgcsLQYAwF09
oGgxBgDAW7ktdy8GAADp7vGkMAYAgEQQES8rBgBAvuec8y0GAMBf9ON5IVgQxJ5VYy8haAyoIgcvIRhIQ2O0LQYAAJyKJEcrBgAA
x5WRL0GADNvyOcYBEAxpb2NJIQBsvvH2ky8GAABoo896MAYAQM5dJDovBgBA8ZGpYQEo9BMClLbVFC4GAEB3bgeXKwYAgAkHJAos
BgDAdNbcES0GAID8XxcoLgYAwN6iSXgtBgBAqVG7Yi0GAADg/GhhLwYAAH0QwYkwBgDAW8I0CS4GAAAAIVcJLQYAgErcLLwtBgAA
gPtgHiwGAABl2O/3LgYAwJQ/988qBgAAiNiOmy0GAAD352EaKwYAgKRze8kwBgCAW4aJ8ioGAIBOH4mxKwYAwGS+ZCkuBgAAzNRT
KSwGAIC8v1IsLgYAQI3jku4wBgBAJM2/Ky0GAIDK0oTfLAYAAL/I5QUxBgCAhECvaysGAICPTPy4LgYAAKotR58xBgBAIsKspjEG
AIDxw1M5LQYAwBn9iNAtBgBAHr7JlysGAMBUVW3cLAYAgKzY4OcrBgAAdVd1ky4GAEA9HpnfMAYAwBXHCBArBgBAS3DLDCsGAMAG
+63+MAYAAHuYvJ0xBgCAjpJSGi0GAMCk8MSkKwYAgIL41hcuBgDAjxctAS0GAABNVjV6LgYAQPvj3OAwBgCA2AkgrS4GAIBB41GV
MQYAwL0RVwktBgDA0Ilpfy0GAIAYtgGJLwYAANuV6lMxBgDAcN9YeCwGAIDN18v6KwYAQJn2gyowBgDA7xFgICsGAMCZjT0dKwYA
AFupfAssBgBA8slqWCsGAACEsiD/LwYAwPeA3eEuBgAA/eJnqioGAAC/j9oiKwYAwBeiM5ItBgAAbizxX0EADNKZ/xgBgJB7HlTy
KgYAgDpSjSowBgBAwlrC4SoGAEC2JVOoLQYAQJEOBkkvIaAIdexoAXgM3Py5QUFg9CoBcGadWC4GAMAPW+f9LwYAALd4KlYrBgBA
o6oCnTAGAAABqBgkLQYAwBnCA5csBgCAAvxk0ioGAIBNasXGMAYAQEbu3eYuBgBAskp3fy8GAECaKTcmLwYAAOmD5nArBgCAef8j
OCwGAMBfMpXOKgYAwIOWnYMwBgAAKLDo5SoGAEDMAnWZLAYAAIrmCJQtBgAAYPQqYS0GAEBVqfdPLQYAABndkGkxBgBAWiHFLC4G
AMB/xtITMAYAAKuZPz8xBgCAoSUlzi0GAAClEZ4DLgYAABUEw7orBgDAtoOxKS4GAMAD6l/QKgYAgPHLAmktBgBASeKJ9y8GAMBS
oH8tKwYAAFbt32QwBgCAPrL7bC8GAMC0DqjUMAYAAKRZzQcxBgAA1jQDKC0GAADehFKhIBAAJMP+mCEoLMRrLzUuBgDA5mL5yyFo
MCTihegwBgBAibAWAjABmCQcA4UuBgBAb63AAShQAAzI4FsvBgBAsnq1kTEGAMCl6KYrASAMA7FSnAEY8FtAcgQTMQYAgNSK71gx
BgBAMxYeyC8GAAACiSrHLAYAALUDjd0wBgAA9YblKS0GAADX3mkeLwYAgBmMYCgtBgCAxgoUJzEGAIBsCC//KwYAACwwHCQrBgBA
mRaengFADFb5kGgBcPCLN0NB5yoGAMBXxfCvKwYAwCfbRi8vBgCA8UD4dywGAIDWq+FRLQYAgM+7M30uBgAA7Vl63S0GAADZn+Tc
LQYAgNpOKpsrBgBAm8VApy8GAACCtnV/KwYAALCYhWgxBgDAC1zGri8GAIDznhLfLAYAwH8+M+ksBgDALjYfQi0GAEBxlF0QLQYA
QDpYUj4h8AwtW0TkAZAsm2nJPiwGAAACDODqAZAMzCYK5AGwCGekN8GgkEBwZjF8KwYAwBt8Na8wBgAAuTdY7y8GAAB6zYSEKwYA
gI8WqW4BEBA820TwKiFIJJdSITAGAEAUHjshMHBATGaItS0GAEDMoDY2KwYAQKqAG2UsBgCAitYUDwEIDDgPBtABcFCTPUx6LAYA
wCrW5m0vBgBAPaHZ6TABUAR2ZqXASOGzvQYtBgDAVC51BysGAACQ89lhWHCALnPuTjEGAIAXzcucLAYAAICuWKcqBgBA9RYGUgEw
CLH4imFYEMBIi2eMAVAMO8GN8QEgDCMonZ4BIPBSsK4e2zAGAMCR9ggyKwYAwO6s2QQuBgCAs6jpYTEGAAAni5ucLQYAwJp3Chcu
BgCA7V1tHy4GAAC6F+X+LwYAQHDwgGsrBgAAPaKruS0GAIBE6mNlMIh8Z8+TLgYAADnpjDkvBgCAKkx9yDAGAACbu54eLAYAQHXe
PQFoNEDQ3wVxLgYAgOyYA+0wwWgk57I+LQYAgJv+TuHYEIAzlGqNARAsyg8v9y0GAACPcuErAZgoFhgFyi4GAED2LGRB+BDACxyE
nwFoqDWGumsuBgCABvXKSzAGAMDLrba3LgYAwJSMRy8uBgCAiy2zzioGAACHRONBqDAAQoet+ywGAACnzKknAWBIqQnm2y8GAMCx
ZlJ2MAYAAEEV0SFQFMCpvrxBLQ7QCCjaMAEwBgCAPLpg6AEoKAJgcpouBgAAI21VIYA0gFS8/TksBgBAH5Z28zAh8AiRhesBKKho
5I/HMAYAALCm720sBgAAmn4O2CwGAEBcCGtELQYAAC9mZZAtBgDAg+MMDmAIEICWJrmfASAopeOaMjEGAIBk9b/B8CxAZayTri4G
AMAAABghwBAAwcQ6dgEoEBVfZlEuQcgIxSIaAVBQ3BHwLS8GAEBxuiF3MQYAwN20N6wxoUgsQkOtKwYAgJyWBWQtgeAIzvz1QRAs
YDjIpSsGAEAyvYuuAQgMH4ZJGAEIDGjNlLQBuDAuwW80LwYAgHV5R54x4eAI+qE4ASAMl2gidAFISIY6pj0uBgBABxhtJCwGAMCz
kMAOuAlQwIS337svBgDADmh2ETEGAMC6ee/aAVAM9iQmAwEwDExvmIQBMEw2Qx/1LQYAwG3tGS4tBgCAsXdVdAEYCKKyeAHgUMDJ
sVxdLAYAAOskYnIuBgCAwUVixwFYbIZ/V+UtBgBAbXr62S8GAEAIFV2mLwYAgMJ+hF0BKBB5+6xKKwF4JPxXLCsGAIDKU0YOmAoQ
gOUeW6EBOAyKjgDdAVhoq6J8PzEGAIDkqGHCLAYAwLbV7+IqBgDABpvcYQhwwJket6grBgBA5ajGSDAGAMAa/2uWMQYAQDoHY6oB
aAgdxKJFQJA2KLcyLwYAwD/vnCwtBgDAghiAWywGAAAWyFLhLQYAgLj1ySUs4RBE5NV8LQYAwHVctvkqBgBAf7JLYRAMAMkQCmFg
LEBXMICPMQYAwGqYaBIwCyw/A4IMLgYAwHHekcwBgCgFDnY+MAYAAEqnPKHAcMA80/pGLgYAQHOYnAgxBgAATKbUAjAGAMBhki52
AWgsh9BrUywGAEA45fitAQjwW/ZTn9UsBgAAmeKv3jAGAEAzFDiVMQYAQJflcToxBgAAPxtQ8CwGAICaDpGeLQYAgNpCwX8vBgDA
vZvcZi4GAIC3aoSNMQYAwHrgOcQwBgDANX5AqS0GAEAxP3NwAYgI6XEWYbBQgIFSjawqBgAAzSUlmS8GAABjD0izAXhwz/VL8i0G
AABkEil5LAYAQMA2+UUuBgCASdWwFCyBSAiZLLABIExLUctQLQYAQLrm3VkxBgAAMWomMgEoCLvJ5g5gCFAAWxd+7CsGAMDhhNXq
MAYAAAYXqQABICx2uBvNKgYAwGG0aXUBGExQ1bUNLwYAQLL2pwswBgDAz+zaagFYEOQpqyAwgdgEks8hiDAANVJ3nSsGAICdf3UR
AQgMDq4alQEIDATlbnMBCAjKAirFOFA9YUSzKgYAgIdVeNorBgBAPqXcoy2BoCSMRB0sBgAAUq1DYUgsAMK2rWIvBgAAX/Kzxejo
zBh3ACwGAEALRIDJKgYAACnh2aIsBgDAmBaXYzAGAADbAZolMAYAgMTOOBItBgCA43JMPTAGAEAWRo1BwDQA1U6tGysGAAAB5Tuo
KsFoLC+u7DAGAIDz18ycMA5wCyjWgTgtBgDAzWMyNwEQkPwaePouBgDAf+uKUzEGAMAHGyO7LgYAQG4M+t4uBgDAY4MC/i4hOEwr
bNMuBgAA1LlmRC8GAEDSJ8MNK2FoDPTQ6CyhoCwIducrBgDACIREuC0BqCRiezwsBgBAixFLDhAMUEBTZK62LgYAgJiRkDEuBgDA
VIU/EgEgSCXUF98tBgDA3DWMsy0GAIBQTyyliAjOfNwBYBBA0Yyq/AEoCOmikQ6wDDDAFYYroDEGAIDO0DtSAUgMKkaP1gF4DN/t
CQMBQAiCLcAO4AgQAOLm06EBIEwPySeKMQYAQAPbW9UsBgCAMHRscQEYTFwHGh0rBgAAqwCZLS0GAABY8miYARAQWaJiHS7hMGjv
mr8sBgDA1ASqeS0GAIBt0gnMLAYAAJZedsghWIgimvnPLgYAQMQmraUwBgBA+7dUZzEGAIBAwTBnMAYAgO9lrqHYTICcMQsVLAYA
QJ1Y2J8tBgDAt2KMoRDQgL890NUrBgBAUP2mjjEGAAA+I09fMAYAwKpb+jArBgAAv+uONCwGAABCPLSVMQYAgJVN5vwBIAh4xcOh
CAwAQ16jJeAQ9XMtfS2BIAi48AwhmAyafsrYAQhIR1t20S4GAIAEBn9BMQYAQBm4WQ4gCzxAqZJeESwGAIBJkIT+KwYAFQAV0AoV
2gosFaQIFRAVBhUGHBgIwN20N6wxBgAYCEAF0iChKgYAFgAoCMDdtDesMQYAGAhABdIgoSoGABERAAAAqAX0pwIDAAAApAgBCn8A
BCDAAAQUYMABCCSgwAIMNODAAxBEIMEEFFRgwQUYZKDBBhx04MEHIIQgwggklGDCCSikoMIKLLTgwgswxCDDDDTUYMMNOOSgww48
9ODDD0AEIcQQRBRhxBFIJKHEEkw04cQTUEQhxRRUVGHFFVhkocUWXHThxRdghCHGGGSUYcYZaKShxhpstOHGG3DEIcccdNRhxx14
5KHHHnz04ccfgAQiyCCEFGLIIYgkosgijDTiyCOQRCLJJJRUYsklmGSiySacdOLJJ6CEIsoopJRiyimopKLKKqy04sorsMQiyyy0
1GLLLbjkossuvPTiyy/ABCPMMMQUY8wxyCSjzDLMNOPMM9BEI8001FRjzTXYZKPNNtx048034IQjzjjklGPOOeiko8467LTjzjvw
xCPPPPTUY889+OSjzz789OPPPwAFJNBABBVk0EEIJaTQQgw15NBDEEUk0UQUVWTRRRhlpNFGHHXk0UcghSTSSCSVZNJJKKWk0kos
teTSSzDFJNNMNNVk00045aTTTjz15NNPQAUl1FBEFWXUUUglpdRSTDXl1FNQRSXVVFRVZdVVWGWl1VZcdeXVV2CFJdZYZJVl1llo
paXWWmy15dZbcMUl11x01WXXXXjlpddefPXl11+ABSbYYIQVZthhiCWm2GKMNebYY5BFJtlklFVm2WWYZabZZpx15tlnoIUm2mik
lWbaaailptpqrLXm2muwxSbbbLTVZtttuOWm22689ebbb8AFJ9xwxBVn3HHIJafccsw159xz0EUn3XTUVWfdddhlp9123HXn3Xfg
hSfeeOSVZ9556KWn3nrstefee/DFJ9989NVn330J+OWn33789efffwAGKOCABBZo4IEIJqjgggw26OCDEEYIAAAAAAAAABUEGUw1
ABgGc2NoZW1hFQYAFQQlAhgHdXNlcl9pZAAVDCUCGAhwbGF0Zm9ybSUATBwAAAAVBCUCGAhsb2dpbl90cyUUTIwSHCwAAAAAABak
CBkcGTwmABwVBBk1AAYQGRgHdXNlcl9pZBUCFqQIFswOFrwLJv4DJggcGAg3AAAAAAAAABgIAQAAAAAAAAAWACgINwAAAAAAAAAY
CAEAAAAAAAAAEREAGSwVBBUAFQIAFQAVEBUCADwpBhkmAKQIAAAAJgAcFQwZNQAGEBkYCHBsYXRmb3JtFQIWpAgWvgMWygMmlgwm
xAscNgAoA3dlYhgHYW5kcm9pZBERABksFQQVABUCABUAFRAVAgA8FpwiGQYZJgCkCAAAACYAHBUEGTUABhAZGAhsb2dpbl90cxUC
FqQIFpZOFpxMJsxPJo4PHBgIwN20N6wxBgAYCEAF0iChKgYAFgAoCMDdtDesMQYAGAhABdIgoSoGABERABksFQQVABUCABUAFRAV
AgA8KQYZJgCkCAAAABagYBakCCYIFqJbABkcGAxBUlJPVzpzY2hlbWEYwAIvLy8vLytnQUFBQVFBQUFBQUFBS0FBd0FCZ0FGQUFn
QUNnQUFBQUFCQkFBTUFBQUFDQUFJQUFBQUJBQUlBQUFBQkFBQUFBTUFBQUNBQUFBQVBBQUFBQVFBQUFDYy8vLy9BQUFCQ2hBQUFB
QWdBQUFBQkFBQUFBQUFBQUFJQUFBQWJHOW5hVzVmZEhNQUFBWUFDQUFHQUFZQUFBQUFBQUlBMFAvLy93QUFBUVVRQUFBQUlBQUFB
QVFBQUFBQUFBQUFDQUFBQUhCc1lYUm1iM0p0QUFBQUFBUUFCQUFFQUFBQUVBQVVBQWdBQmdBSEFBd0FBQUFRQUJBQUFBQUFBQUVD
RUFBQUFDQUFBQUFFQUFBQUFBQUFBQWNBQUFCMWMyVnlYMmxrQUFnQURBQUlBQWNBQ0FBQUFBQUFBQUZBQUFBQQAYIHBhcnF1ZXQt
Y3BwLWFycm93IHZlcnNpb24gMjUuMC4xGTwcAAAcAAAcAAAAFwMAAFBBUjE=
""",
}

_data_dir = pathlib.Path("data") / "024-monthly-active-users"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

logins = spark.read.parquet((_data_dir / "logins.parquet").resolve().as_posix())
logins.createOrReplaceTempView("logins")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'login_month': '88bd060b72cd23d81d9aafac1bb1b8246de4171f48023c2a7679d5933516c4ba',
                         'monthly_active_users': '47c336a291d0d7c8e7c24e6f7e301996e7992225c9f3ef0e6a7eb21abac014b9',
                         'peak_daily_active_users': '5b38f97bf823217cdb11298b3a8298afd12d701f70b83f161443b0bb2e62765a',
                         'platform': '8f8d9694803ebc21145c5084fe3debb9ab24af216ab43de0e9c37caf7674802b',
                         'total_logins': '158ed04f39767642e3fbe1a58089a02053c26e9d0dfa63fdda700157ca036f9e'},
 'columns': [['login_month', 'date'],
             ['monthly_active_users', 'bigint'],
             ['peak_daily_active_users', 'bigint'],
             ['platform', 'string'],
             ['total_logins', 'bigint']],
 'fingerprint': 'd23c77dd60caed7ddb1b77412e8905b602b56b2a6de0fd8e66e624d9116b1800',
 'order_matters': False,
 'row_count': 9}

check = make_check(EXPECTED)

## The situation

The growth dashboard shows monthly active users (MAU) per platform. A daily active
users table already existed, so the first version of the dashboard added the daily
numbers up to get the month.

Every monthly number came out more than twice too high. A user who logs in on
twenty days is twenty daily active users and one monthly active user.

## Input tables

### `logins`

One row per login. A user can log in several times a day, and on more than one
platform.

| Column | Type | Description |
|---|---|---|
| `user_id` | BIGINT | The user. |
| `platform` | STRING | `web`, `android` or `ios`. |
| `login_ts` | TIMESTAMP_NTZ | When the login happened. |

## Requirement

Return one row per month and platform that has logins.

- `login_month` is the first day of the calendar month of the login. A login at
  23:59:59 on 31 January belongs to January; one second later belongs to February.
- `monthly_active_users` is the number of different users with at least one login on
  that platform in that month.
- `total_logins` is the number of logins on that platform in that month.
- `peak_daily_active_users` is the highest number of different users on that platform
  on any single day of that month.

## Expected output

One row per month and platform.

| Column | Type | Description |
|---|---|---|
| `login_month` | DATE | First day of the month. |
| `platform` | STRING | |
| `monthly_active_users` | BIGINT | Distinct users in the month. |
| `total_logins` | BIGINT | |
| `peak_daily_active_users` | BIGINT | Highest distinct users on one day. |

In [ ]:
logins.show(5)

## Your answer

Write one Spark SQL query. The tables are available as views: `logins`.

In [ ]:
query = """
-- Write your Spark SQL query here.
"""
result = spark.sql(query)
result.show()

In [ ]:
check(result)